In [11]:
# ── 1. Repos & deps ──────────────────────────────────────────────────────────
!git clone https://github.com/goombalab/hnet.git
!git clone --recurse-submodules https://github.com/Chaoqi-LIU/oat.git
!rm -rf congenial-adventure && git clone https://github.com/SDogya/congenial-adventure.git && cp -r congenial-adventure/. . && rm -rf congenial-adventure

import os
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('wandb')

!uv add "zarr<3.0.0" dill einops numba vector-quantize-pytorch accelerate huggingface_hub "robomimic<0.3.0" torchvision wrapt pillow pandas diffusers
!uv sync

# Patch OAT's lr_scheduler.py: newer diffusers removed Union/Optional/Optimizer exports
p = 'oat/oat/model/common/lr_scheduler.py'
txt = open(p).read()
marker = 'from diffusers.optimization import ('
if marker in txt and 'from typing import Union' not in txt:
    idx = txt.index(marker)
    end_idx = txt.index(')', idx) + 1
    header = (
        'from typing import Union, Optional\n'
        'from torch.optim import Optimizer\n'
        'from diffusers.optimization import SchedulerType, TYPE_TO_SCHEDULER_FUNCTION'
    )
    open(p, 'w').write(txt[:idx] + header + txt[end_idx:])
    print('lr_scheduler.py patched OK')
else:
    print('lr_scheduler.py already clean, skipping')

# Patch OAT: заглушить print'ы про нормализацию
for path in [
    'oat/oat/perception/state_encoder.py',
    'oat/oat/perception/robomimic_vision_encoder.py',
]:
    txt = open(path).read()
    txt = txt.replace(
        'print(warning_msg(f"no normalizer params for port {port}, skipping normalization."))',
        'pass  # suppressed'
    ).replace(
        'print(f"no normalizer params for port {port}, skipping normalization.")',
        'pass  # suppressed'
    )
    open(path, 'w').write(txt)
print('normalizer prints suppressed OK')


fatal: destination path 'hnet' already exists and is not an empty directory.
fatal: destination path 'oat' already exists and is not an empty directory.
Cloning into 'congenial-adventure'...
remote: Enumerating objects: 475, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 475 (delta 18), reused 29 (delta 10), pack-reused 431 (from 2)
Receiving objects: 100% (475/475), 81.61 MiB | 45.54 MiB/s, done.
Resolving deltas: 100% (208/208), done.
Resolved 118 packages in 89ms                                        
Audited 115 packages in 1ms                                          
Resolved 118 packages in 0.87ms
Audited 115 packages in 1ms
lr_scheduler.py already clean, skipping
normalizer prints suppressed OK


In [4]:
!ls -ld /kaggle/working/oat/data/libero/*

-rw-r--r-- 1 root root 3528193175 Apr 13 12:03 /kaggle/working/oat/data/libero/libero10_N500.zarr.zip
-rw-r--r-- 1 root root         24 Apr 13 12:03 /kaggle/working/oat/data/libero/README.md


In [2]:
# ── 2. Dataset (скачиваем прямо туда, куда смотрит OAT по дефолту) ───────────
import os
from huggingface_hub import snapshot_download
from huggingface_hub import login
hf_token = UserSecretsClient().get_secret('hugface')

if hf_token:
    login(token=hf_token)
else:
    print("Ошибка: Секрет 'hugface' не найден.")

In [3]:

from huggingface_hub import snapshot_download
os.makedirs('/kaggle/working/oat/data/libero', exist_ok=True)
snapshot_download(
    repo_id='chaoqi-liu/libero10_N500.zarr',
    repo_type='dataset',
    local_dir='/kaggle/working/oat/data/libero'
)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

'/kaggle/working/oat/data/libero'

In [5]:
# Распаковываем архив прямо в ту же папку
!unzip -o -q /kaggle/working/oat/data/libero/libero10_N500.zarr.zip -d /kaggle/working/oat/data/libero/

# Проверяем, что папка появилась
!ls -ld /kaggle/working/oat/data/libero/*

drwxrwxr-x 4 root root       4096 Jan 23 06:29 /kaggle/working/oat/data/libero/libero10_N500.zarr
-rw-r--r-- 1 root root 3528193175 Apr 13 12:03 /kaggle/working/oat/data/libero/libero10_N500.zarr.zip
drwxr-xr-x 3 root root       4096 Apr 13 12:04 /kaggle/working/oat/data/libero/__MACOSX
-rw-r--r-- 1 root root         24 Apr 13 12:03 /kaggle/working/oat/data/libero/README.md


In [ ]:
# ── 3. Train OAT tokenizer (~2-3h, 300 epochs) ───────────────────────────────
!uv run python oat/scripts/run_workspace.py \
    --config-name=train_oattok \
    task/tokenizer=libero/libero10 \
    training.num_epochs=300 \
    logging.project=VLA-experiment \
    task.tokenizer.dataset.zarr_path="/kaggle/working/oat/data/libero/libero10_N500.zarr"

In [ ]:

# ── 4. Train FD-DRAT ─────────────────────────────────────────────────────────
import glob
ckpt_candidates = sorted(
    glob.glob('/kaggle/working/output/**/model.ckpt', recursive=True) +
    glob.glob('/kaggle/working/**/model.ckpt', recursive=True)
)
TOK = ckpt_candidates[-1] if ckpt_candidates else ''
print(f'Using tokenizer: {TOK}')

!HYDRA_FULL_ERROR=1 MPLBACKEND=agg uv run run.py \
    strategy=single_gpu \
    model.tokenizer_ckpt={TOK} \
    dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr \
    batch_size=16


Using tokenizer: /kaggle/working/src/model/model.ckpt
Seed set to 42
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
[W413 12:16:55.762085280 Utils.hpp:137] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
[W413 12:16:55.772271632 Utils.hpp:137] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. St